# Non-Parametric Hypothesis Testing — Cheat Sheet

A fast reference for choosing, running, and interpreting the seven tests
covered in this lab. Keep this open while you work on the lab or a real
project — it's meant to be scanned, not read cover to cover.

## 1. Decision Tree — "Which test do I use?"

```
How many groups / variables am I comparing?

├── ONE sample vs a hypothesized value, or paired/matched samples (same subjects,
│   two conditions)
│   └── Continuous/ordinal data, normality doubtful
│       -> WILCOXON SIGNED-RANK TEST (stats.wilcoxon)
│
├── TWO independent groups
│   └── Continuous/ordinal data, normality and/or equal-variance doubtful
│       -> MANN-WHITNEY U / RANK-SUM TEST (stats.mannwhitneyu)
│   └── Very small samples, want an exact/distribution-free p-value for an
│       arbitrary statistic (mean diff, median diff, correlation, etc.)
│       -> PERMUTATION TEST (stats.permutation_test)
│
├── THREE OR MORE independent groups
│   └── Continuous/ordinal data, normality/equal-variance doubtful
│       -> KRUSKAL-WALLIS TEST (stats.kruskal), then pairwise
│          Mann-Whitney + Bonferroni correction if you need to know WHICH pairs differ
│
├── CATEGORICAL data — counts / frequencies
│   ├── ONE categorical variable, comparing observed counts to expected counts
│   │   -> CHI-SQUARE GOODNESS-OF-FIT (statsmodels chisquare / scipy chisquare)
│   └── TWO categorical variables, testing if they're related
│       -> CHI-SQUARE TEST OF INDEPENDENCE (scipy chi2_contingency)
│
└── TWO variables, want to know if they move together (monotonically)
    └── Ordinal data, or continuous but non-linear/has outliers
        -> SPEARMAN'S RANK CORRELATION (stats.spearmanr)
```

**Golden rule from the source material:** *if a parametric test's assumptions
are genuinely satisfied, use it* — non-parametric tests trade some statistical
power for fewer assumptions, they are not a free lunch.

## 2. At-a-Glance Table

| Test | Parametric analogue | Compares | Function | Assumptions kept |
|---|---|---|---|---|
| Permutation test | t-test / any | any statistic, 2 groups | `scipy.stats.permutation_test` | independence only |
| Mann-Whitney U (rank-sum) | independent t-test | distributions of 2 independent groups | `scipy.stats.mannwhitneyu` | independence, similar shape (for "location shift" interpretation) |
| Wilcoxon signed-rank | paired t-test | paired/matched differences | `scipy.stats.wilcoxon` | independence of pairs, symmetric differences |
| Kruskal-Wallis | one-way ANOVA | medians of ≥3 independent groups | `scipy.stats.kruskal` | independence, similar shape across groups |
| Chi-square goodness-of-fit | — | observed vs expected counts, 1 variable | `statsmodels.stats.gof.chisquare` | independent observations, expected counts not too small |
| Chi-square independence | — | association between 2 categorical variables | `scipy.stats.chi2_contingency` | independent observations, expected counts ≥ ~5 |
| Spearman's rho | Pearson correlation | monotonic association, 2 variables | `scipy.stats.spearmanr` | independence of pairs |

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 3. Permutation Test — Code Template

```python
def statistic_function(set_one, set_two):
    return np.mean(set_one) - np.mean(set_two)   # swap in median, correlation, etc.

result = stats.permutation_test(
    (sample_1, sample_2),
    statistic_function,
    alternative='two-sided',   # or 'less' / 'greater' — think about direction!
    random_state=rng,
)
print(result.statistic, result.pvalue)
plt.hist(result.null_distribution, bins=30)
plt.axvline(result.statistic, color='red')
```

**Notes**
- Works for *any* test statistic — mean diff, median diff, correlation,
  variance ratio, whatever you can code.
- Exhaustive permutation is only feasible for small n (n! or C(n, k) blows up
  fast — see growth table below). `scipy` switches to Monte Carlo resampling
  automatically for larger n (`n_resamples`, default 9999).
- p-value = proportion of permuted statistics at least as extreme as observed
  (+ a small correction so p is never exactly 0).

In [2]:
# Permutation growth reference
sizes = np.arange(1, 13)
import scipy as sp
pd.DataFrame({'n': sizes, 'n!': sp.special.factorial(sizes)})

,n,n!
0,1,1.0
1,2,2.0
2,3,6.0
3,4,24.0
4,5,120.0
5,6,720.0
6,7,5040.0
7,8,40320.0
8,9,362880.0
9,10,3628800.0


## 4. Mann-Whitney U / Rank-Sum — Code Template

```python
H0 = "The two groups come from the same distribution."
Ha = "The distributions differ in location (or use alternative='greater'/'less')."

result = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')
print(result)

# Effect size: rank-biserial correlation
n1, n2 = len(group_a), len(group_b)
U = result.statistic
U_other = n1 * n2 - U
r_rb = 1 - (2 * min(U, U_other)) / (n1 * n2)
print("rank-biserial r =", r_rb)
```

**Manual mechanics (what's happening under the hood):**
1. Pool both groups, rank all values (average ranks for ties).
2. T = sum of ranks in one group.
3. `Z = (T - mean(T)) / sd(T)`, where
   `mean(T) = n1 * (n1+n2+1) / 2` and
   `sd(T) = sqrt(n1*n2*(n1+n2+1)/12)` (before tie correction; the lab notebook
   uses an equivalent pooled-size form from the corrected ranks).
4. p-value from the normal distribution (`scipy.stats.norm`), or exact for
   very small samples (`method='exact'` in `mannwhitneyu`).

**Effect size cutoffs (rank-biserial r, same scale as Cohen's d loosely):**
~0.1 small · ~0.3 medium · ~0.5 large.

## 5. Wilcoxon Signed-Rank — Code Template

```python
H0 = "Median of paired differences = 0."
Ha = "Median difference != 0 (or use alternative='greater'/'less')."

result = stats.wilcoxon(before, after, alternative='two-sided')
print(result)

diffs = before - after
diffs = diffs[diffs != 0]                       # drop zero differences
ranks = pd.Series(np.abs(diffs)).rank(method='average').values
S_pos = ranks[diffs > 0].sum()
S_neg = ranks[diffs < 0].sum()
n = len(diffs)

# Effect size: matched-pairs rank-biserial correlation
r_matched = (S_pos - S_neg) / (n * (n + 1) / 2)
print("matched-pairs rank-biserial r =", r_matched)

# Manual normal-approximation p-value (for n large enough, no ties)
mean_S = n * (n + 1) / 4
sd_S = np.sqrt(n * (n + 1) * (2*n + 1) / 24)
Z = (S_pos - mean_S) / sd_S
print("Z =", Z, " one-sided p =", stats.norm.sf(Z))
```

**Watch out for:** `scipy.stats.wilcoxon` uses an exact method by default for
small, tie-free samples and a normal approximation otherwise — results may
differ slightly from a hand calculation using only the normal approximation.

## 6. Kruskal-Wallis — Code Template

```python
H0 = "All group medians are equal."
Ha = "At least one group's median differs."

result = stats.kruskal(group1, group2, group3)   # any number of groups >= 2
print(result)

# Effect size (eta-squared-like)
H, k, n = result.statistic, 3, len(group1) + len(group2) + len(group3)
eta_sq = (H - k + 1) / (n - k)
print("eta^2-like =", eta_sq)

# Post-hoc: pairwise Mann-Whitney + Bonferroni, only if the omnibus test rejects H0
from itertools import combinations
groups = {'g1': group1, 'g2': group2, 'g3': group3}
n_comparisons = len(list(combinations(groups, 2)))
for a, b in combinations(groups, 2):
    stat, p = stats.mannwhitneyu(groups[a], groups[b])
    print(a, b, "raw p =", p, "Bonferroni p =", min(p * n_comparisons, 1.0))
```

**Effect size guide (eta-squared-like):** 0.01 small · 0.06 medium · 0.14 large
(same convention as ANOVA eta-squared).

## 7. Chi-Square Goodness-of-Fit — Code Template

```python
from statsmodels.stats.gof import chisquare
from scipy.stats import chi2

H0 = "Observed frequencies match the expected/hypothesized frequencies."
Ha = "At least one category's frequency differs from expected."

chi_stat, p_value = chisquare(f_obs=[45, 30, 15], f_exp=[30, 30, 30])
crit = chi2.ppf(0.95, df=len([45,30,15]) - 1)   # df = k - 1
print(chi_stat, p_value, crit)

n = sum([45, 30, 15])
cohens_w = np.sqrt(chi_stat / n)   # effect size
print("Cohen's w =", cohens_w)     # 0.1 small, 0.3 medium, 0.5 large

# Power / sample-size planning
from statsmodels.stats.power import GofChisquarePower
from statsmodels.stats.gof import chisquare_effectsize
es = chisquare_effectsize(probs0=[1/3,1/3,1/3], probs1=[0.45,0.30,0.25], cohen=True)
n_needed = GofChisquarePower().solve_power(es, power=0.8, alpha=0.05, n_bins=3)
print("n needed for power 0.8:", n_needed)
```

**Rule of thumb:** every expected cell count should be ≥ ~5 (some say ≥10 for
small df) or the chi-square approximation to the p-value gets unreliable.
This test is always one-tailed / right-tailed in the sense that only large χ²
values indicate a departure from H0.

## 8. Chi-Square Test of Independence — Code Template

```python
from scipy.stats import chi2_contingency, chi2

H0 = "The two categorical variables are independent."
Ha = "The two categorical variables are associated."

observed = np.array([[1429, 1235], [1216934, 22663]])   # rows x cols
chi_stat, p_value, dof, expected = chi2_contingency(observed, correction=False)
print(chi_stat, p_value, dof)
print(expected)

# Yates' correction (recommended mainly for small expected counts, e.g. 2x2 tables
# with cells < ~10; many practitioners skip it for large samples)
chi_stat_y, p_value_y, dof_y, _ = chi2_contingency(observed, correction=True)

# Effect size: Cramer's V
n = observed.sum()
r, c = observed.shape
cramers_v = np.sqrt(chi_stat / (n * min(r-1, c-1)))
print("Cramer's V =", cramers_v)   # 0.1 small, 0.3 medium, 0.5 large (for df_min=1)
```

**Reminder:** with huge sample sizes, chi-square (and most hypothesis tests)
will find "statistically significant" associations that are practically tiny.
Always report Cramer's V (or another effect size) alongside the p-value.

## 9. Spearman's Rank Correlation — Code Template

```python
H0 = "rho = 0 (no monotonic association)."
Ha = "rho != 0."

rho, p_value = stats.spearmanr(x, y)
print(rho, p_value)

# Bootstrap CI (useful for small n where the asymptotic p-value is shaky)
n_boot = 2000
boot = np.empty(n_boot)
x_arr, y_arr = np.array(x), np.array(y)
n = len(x_arr)
for i in range(n_boot):
    idx = rng.integers(0, n, size=n)
    boot[i] = stats.spearmanr(x_arr[idx], y_arr[idx]).correlation
ci = np.nanpercentile(boot, [2.5, 97.5])
print("95% bootstrap CI:", ci)
```

**rho interpretation (same convention as Pearson r):**
~0.1 weak · ~0.3-0.5 moderate · ~0.7+ strong.
Spearman is Pearson's r computed on the **ranks** of the data instead of the
raw values — that's why it only detects monotonic relationships, not just
linear ones, and why it's robust to outliers.

## 10. Common Pitfalls (all tests)

- **Wrong `alternative` direction.** Always double-check which group/condition
  corresponds to "greater" vs "less" before you interpret a one-sided p-value.
- **Ties.** All of these rank-based tests handle ties via *average ranks*, but
  heavy tie counts reduce power and can invalidate the normal-approximation
  variance formulas — prefer exact/permutation methods when there are many
  ties and small n.
- **p-value ≠ effect size.** A tiny p-value with a huge sample can still mean a
  trivial real-world effect (see Section 8's crash-data example — n > 1
  million). Always pair a test with an effect size.
- **Independence is never optional.** None of these tests relax the
  independence assumption — if your samples are clustered, repeated-measures
  without pairing, or otherwise dependent, none of the tests above are valid
  without modification (e.g., need a mixed-effects or clustered-permutation
  approach instead).
- **"Non-parametric" does not mean "assumption-free."** Mann-Whitney assumes
  similar shape/spread across groups if you want to interpret it as a shift in
  location; Kruskal-Wallis has the same caveat with 3+ groups.
- **Small samples still need caution.** Wide confidence/bootstrap intervals
  (see Part 7 of the lab) are a feature, not a bug — they tell you honestly how
  much you don't know.